<a href="https://colab.research.google.com/github/arnavm2k3/TorchCode-Solutions/blob/main/templates/12_linear_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔴 Hard: Linear Self-Attention

Implement **Linear Attention** — O(S·D²) instead of O(S²·D), enabling efficient long-sequence processing.

Replace softmax with a **kernel feature map** $\phi$:

$$\text{LinearAttn}(Q,K,V) = \frac{\phi(Q) \left(\phi(K)^T V\right)}{\phi(Q) \cdot \sum \phi(K)}$$

### Feature map
Use $\phi(x) = \text{elu}(x) + 1$ (ensures non-negative features).

### Signature
```python
def linear_attention(Q, K, V):
    # Q: (B, S, D_k), K: (B, S, D_k), V: (B, S, D_v)
    # Returns: (B, S, D_v)
```

### Key insight
Instead of computing the $S \times S$ attention matrix, compute $\phi(K)^T V$ first (a $D_k \times D_v$ matrix), then multiply by $\phi(Q)$.

### Rules
- Must use a feature map (NOT softmax)
- Must be O(S·D²) — should run fast on long sequences
- You **may** use `F.elu`

### References
Linear Attention Fundamentals - [Link](https://haileyschoelkopf.github.io/blog/2024/linear-attn/)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.4 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn.functional as F

In [21]:
# ✏️ YOUR IMPLEMENTATION HERE

def linear_attention(Q, K, V):
    pQ, pKT = F.elu(Q), F.elu(K).transpose(1, 2)

    return torch.divide(torch.bmm(pQ, torch.bmm(pKT, V)), torch.bmm(pQ, torch.sum(pKT, dim=-1, keepdim=True)))

In [22]:
# 🧪 Debug
Q = torch.randn(1, 8, 16)
K = torch.randn(1, 8, 16)
V = torch.randn(1, 8, 32)
out = linear_attention(Q, K, V)
print("Output shape:", out.shape)   # (1, 8, 32)
print("Has NaN?", torch.isnan(out).any().item())

Output shape: torch.Size([1, 8, 32])
Has NaN? False


In [23]:
from torch_judge import check
check('linear_attention')


🧪 Testing: Linear Self-Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (3.7ms)
  ✅ [2/4] No NaN or Inf (2.9ms)
  ✅ [3/4] Gradient flow (1.3ms)
  ✅ [4/4] Runs fast on long sequences (linear complexity) (28.1ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (36.0ms total)
  Progress saved. Run status() to see your dashboard.

